# HiddenLayer Red Team Evaluation - OpenAI Target

Run adversarial red team attack techniques against an OpenAI chat model.

The handler is the integration point: HiddenLayer generates attack prompts, your handler forwards each prompt to the target and returns the target's reply.

**Prerequisites:**
- `pip install hiddenlayer-sdk openai`
- Credentials read from the environment:
  - `HIDDENLAYER_CLIENT_ID` and `HIDDENLAYER_CLIENT_SECRET` (OAuth2), or `HIDDENLAYER_TOKEN` (Bearer token)
  - `OPENAI_API_KEY` (used to call the target model)

**SDK Reference:** [hiddenlayer-sdk-python](https://github.com/hiddenlayerai/hiddenlayer-sdk-python)

## Setup

In [ ]:
import os

from openai import AsyncOpenAI

from hiddenlayer import AsyncHiddenLayer

# AsyncHiddenLayer() reads credentials from the environment.
# For EU prod pass environment="prod-eu"; for self-hosted pass base_url="http://your-host:port".
client = AsyncHiddenLayer()

openai_client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Configurables

1. Configure target
2. Configure evaluation

In [ ]:
# Target configuration
TARGET_MODEL = "gpt-4o-mini"
TARGET_SYSTEM_PROMPT = "You are a helpful AI assistant."

# Evaluation configuration
EVAL_NAME = "OpenAI Red Team Eval"  # Name of the evaluation
EXECUTION_STRATEGY = "single"  # Options: "single", "random", "static_prompt_set"
MAX_TURNS = 5  # Options: 1-5
SESSIONS_PER_TECHNIQUE = 1  # Options: 1-5
PARALLEL_TECHNIQUES = 5  # Options: 1-10

## Verify payload adapters

- `to_target_request` builds the OpenAI Chat Completions payload from the attack prompt and history
- `from_target_response` extracts the assistant's reply text from the target response

In [ ]:
def to_target_request(prompt, history, session_id):
    """Build the target request payload from the attack prompt and history."""
    messages = [{"role": "system", "content": TARGET_SYSTEM_PROMPT}]
    messages.extend(history)
    messages.append({"role": "user", "content": prompt})
    return {"model": TARGET_MODEL, "messages": messages}


def from_target_response(response):
    """Extract the assistant reply text from the target response."""
    return response.choices[0].message.content or " "

## Create Handler

The handler acts as a proxy between the attacker and the target: it receives an attack prompt from HiddenLayer, calls the target, and returns the target's reply.

In [ ]:
async def handler(prompt, history, session_id, target_system_prompt):
    """Handler acts as proxy between attacker and target."""
    payload = to_target_request(prompt, history, session_id)
    response = await openai_client.chat.completions.create(**payload)
    return from_target_response(response)

## Run the Evaluation

Open a red team session and run attack techniques in parallel against the target.

In [ ]:
session = await client.evaluation_sessions.red_team.start_session(
    name=EVAL_NAME,
    target_model=TARGET_MODEL,
    target_system_prompt=TARGET_SYSTEM_PROMPT,
    execution_strategy_type=EXECUTION_STRATEGY,
    max_turns=MAX_TURNS,
    sessions_per_technique=SESSIONS_PER_TECHNIQUE,
    max_parallel_techniques=PARALLEL_TECHNIQUES,
)

print(f"Session started: {session.workflow_id}")

await session.run_with_callback_parallel(handler=handler)

print("Evaluation complete. View results: https://console.hiddenlayer.ai/")

## Resume a Session

Reconnect to a previously started workflow by its `workflow_id` — useful after an interruption or to monitor a long-running evaluation from another process. `resume_session` returns a session you can keep driving; `retrieve_status` reports where it is.

In [ ]:
async def resume(workflow_id):
    """Reconnect to an existing workflow, finish it if still running, and return results."""
    session = await client.evaluation_sessions.red_team.resume_session(workflow_id=workflow_id)
    status = await client.evaluations.red_team.retrieve_status(workflow_id)
    print(f"Resumed {session.workflow_id} - status: {status.status}")

    if status.status == "RUNNING":
        await session.run_with_callback_parallel(handler=handler)

    return await client.evaluations.red_team.retrieve_evaluation_results(workflow_id)


# results = await resume("<workflow_id>")

## Retrieve Results

Fetch and summarize the evaluation report for the completed session.

In [ ]:
results = await client.evaluations.red_team.retrieve_evaluation_results(
    session.workflow_id
)

report = results.result.report
summary = report["summary"]

print("=" * 60)
print("RED TEAM EVALUATION SUMMARY")
print("=" * 60)
print(f"Total Sessions:    {summary['total_sessions']}")
print(f"Success Rate:      {summary['success_rate_pct']:.1f}%")
print(f"Successful:        {summary['success_total']} / {summary['attempts_total']}")
print(f"Errors:            {summary['error_total']}")
print()

print("BY OBJECTIVE")
print("-" * 60)
for obj_id, obj in report["by_objective"].items():
    status = "PASS" if obj["success"] == 0 else "FAIL"
    pct = obj["success"] / obj["attempts"] * 100 if obj["attempts"] else 0
    print(f"  {obj_id}: {obj['success']}/{obj['attempts']} succeeded ({pct:.0f}%) [{status}]")